In [ ]:
import os

import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Suppress non-critical FutureWarning messages from libraries like Seaborn/Pandas
warnings.filterwarnings("ignore", category=FutureWarning)

# Set global Seaborn theme for clean data visualizations
sns.set_theme(style="whitegrid")

# Configure Matplotlib default parameters: high DPI for crisp plots & fix minus sign display issues
plt.rcParams["figure.dpi"] = 100
plt.rcParams["axes.unicode_minus"] = False

# Configure Pandas display options for better DataFrame readability in terminal/notebooks
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
# Format floats with commas and 3 decimal places
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
# Define output and input file paths
FIG_DIR = "../outputs/figures"
CSV_PATH = "../data/raw/Divar.csv"

# Ensure the output directory exists; create it if it doesn't
os.makedirs(FIG_DIR, exist_ok=True)

# Load dataset efficiently without guessing column data types upfront
df = pd.read_csv(CSV_PATH, low_memory=False)

# Output dataset dimensions with comma-formatted row count
print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

In [ ]:
def fix_mojibake(text):
    """
    Repair Persian text corrupted by encoding mismatch (Mojibake).

    Fixes strings originally encoded in UTF-8 but mistakenly read or saved
    using Windows-1256 (cp1256).
    """
    # Bypass non-string entries (e.g., NaN, nulls, or numeric values)
    if not isinstance(text, str):
        return text

    # Fast heuristic check: UTF-8 bytes decoded as cp1256 commonly introduce
    # these specific Persian characters ('ط', 'ظ', 'غ'). If none are present,
    # skip processing to save execution time.
    if ("ط" not in text) and ("ظ" not in text) and ("غ" not in text):
        return text

    try:
        # Re-encode back to raw bytes using cp1256, then properly decode as UTF-8
        fixed = text.encode("cp1256").decode("utf-8", errors="strict")
        return fixed if fixed else text
    except (UnicodeEncodeError, UnicodeDecodeError):
        # Fall back to the original text if byte conversion fails
        return text


# Identify all object/string columns in the DataFrame
text_columns = df.select_dtypes(include="object").columns

# Apply the Mojibake repair function across all identified text columns
for column in text_columns:
    df[column] = df[column].apply(fix_mojibake)

In [ ]:
print(list(df.columns))

df.info()

print(df.dtypes.value_counts())

cols_to_check = [col for col in df.columns if col != "Unnamed: 0"]
real_duplicates = df.duplicated(subset=cols_to_check).sum()
print(f"\nReal duplicate rows: {real_duplicates:,}")

In [ ]:
df.head(20)

In [ ]:
print(df.describe(include="all").T)

In [ ]:
FINANCIAL_COLS = [
    "price_mode",
    "price_value",
    "credit_mode",
    "credit_value",
    "rent_mode",
    "rent_value",
    "rent_type",
    "rent_to_single",
    "rent_credit_transform",
    "transformable_price",
    "transformable_credit",
    "transformable_rent",
    "transformed_credit",
    "transformed_rent",
    "rent_price_on_regular_days",
    "rent_price_on_special_days",
    "rent_price_at_weekends",
]

# Filter to only existing columns to prevent KeyError
target_cols = [c for c in FINANCIAL_COLS if c in df.columns]

# Compute and sort missingness rates across all financial features
missing_df = pd.DataFrame(
    {
        "missing_count": df[target_cols].isna().sum(),
        "missing_pct": df[target_cols].isna().mean() * 100,
    }
).sort_values("missing_pct", ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(x="missing_pct", y=missing_df.index, data=missing_df, palette="crest")
plt.title("Missing Percentage in 17 Financial Columns")
plt.xlabel("Missing Percentage (%)")
plt.tight_layout()
plt.show()


### Missing Data Patterns in Financial Features

Missing values follow platform business logic rather than random loss:

* **Sales Layer (`price_*`)**: ~43% missing $\rightarrow$ ~57% of listings are property sales.
* **Core Rental Layer (`credit_*`, `rent_*`)**: ~65% missing $\rightarrow$ ~35% of listings are long-term rentals.
* **Rent Conversion Layer (`transformed_*`)**: 90–93% missing $\rightarrow$ Applies only to convertible rental listings (~7–10%).
* **Daily Rentals & Flags (`rent_price_on_*`, `rent_to_single`)**: >98% missing $\rightarrow$ Negligible market share on the platform.

#### Key Decisions
1. **Drop Layer 4**: Remove columns with >98% missingness from general models.
2. **Split Datasets**: Create two distinct pipelines for **Sales** (`price_value`) and **Rentals** (`credit_value` / `rent_value`).


In [ ]:
numeric_financials = [
    "price_value",
    "credit_value",
    "rent_value",
    "transformed_credit",
    "transformed_rent",
    "rent_price_on_regular_days",
    "rent_price_on_special_days",
    "rent_price_at_weekends",
]

# Ensure only existing numeric columns are selected to avoid KeyError
num_cols_to_plot = [c for c in numeric_financials if c in df.columns]

# Measure distribution asymmetry to identify heavy-tailed or distorted columns
print("Skewness of financial columns:")
print(df[num_cols_to_plot].skew())

# Use log-scaled y-axis to reveal distribution shape despite extreme outlier compression
df[num_cols_to_plot].hist(bins=30, figsize=(15, 10), log=True)
plt.suptitle("Distribution of Financial Numeric Columns (Log Scale)")
plt.tight_layout()
plt.show()

### Financial Columns Distribution Analysis & Outlier Detection

#### Key Findings
1. **Severe X-Axis Distortion & Extreme Outliers**:
   * Raw pricing columns (`price_value`, `credit_value`, `rent_value`) exhibit severe extreme values reaching $10^{14}$ to $10^{15}$ (e.g., placeholder patterns like `111,111,111,111,111` or `200,250,300,400,450`). These represent user keyboard-mashing, dummy test entries, or system artifacts rather than genuine market transactions.
2. **Extreme Positive Skewness & Left-Side Compression**:
   * Skewness values are exceptionally high across raw metrics (`price_value`: ~112, `credit_value`: ~178, `rent_value`: ~201). The vast majority of legitimate listings are tightly compressed near the lower end of the axis, with the visual distribution completely masked by the extreme right tail.
3. **Controlled Upper Bound in `transformed_rent`**:
   * With a skewness of ~17.3, `transformed_rent` is considerably less distorted than raw fields. Being platform-calculated rather than user-entered, its maximum values are naturally capped near $3 \times 10^9$, avoiding catastrophic keyboard artifacts.
4. **Lower-Bound Anomalies (Zero & Near-Zero Values)**:
   * The distribution issue is two-sided: `rent_value` contains 59,241 zero values (representing full-mortgage/رهن کامل deals), while `price_value` contains 1,902 zero entries (primarily unpriced/توافقی listings encoded as 0).

---

#### Next Steps: Outlier & Boundary Handling
* **Domain-Informed Upper Capping (Beyond Plain 99th Percentile)**:
  * In a 1-million-row dataset, the top 1% corresponds to 10,000 listings. Consequently, relying solely on `.quantile(0.99)` risks retaining multi-billion dummy artifacts.
  * Inspect the upper percentiles granularly (e.g., `np.linspace(0.95, 1.0, 11)`) and establish strict, domain-driven ceiling thresholds (e.g., realistic property caps per market tier).
* **Lower-Bound & Zero-Value Segmentation**:
  * Filter out uninformative symbolic values (e.g., non-positive prices or nominal values under minimal realistic thresholds).
  * Explicitly separate zero-rent listings (`rent_value == 0`) into dedicated rental/mortgage categories rather than treating them as standard numeric distributions.


In [ ]:
plt.figure(figsize=(14, 8))

# Plot horizontal boxplots to visualize distributions and outliers
sns.boxplot(data=df[num_cols_to_plot], orient="h", palette="Set2")

# Apply log scale to handle extreme positive skewness
plt.xscale("log")
plt.title("Boxplot of Financial Numeric Columns (Log Scale)")
plt.xlabel("Value (Log Scale)")
plt.show()

### Financial Features Distribution & Outlier Analysis (Boxplot Summary)

The logarithmic boxplot reveals extreme right-skewness and significant input anomalies across all financial columns, confirming the need for robust data cleaning and transformations before model training.

#### Key Observations
* **`price_value`**: Exhibits the highest baseline magnitude alongside dense, extreme right-tail outliers—primarily caused by unit confusion (Rials vs. Tomans) and placeholder data.
* **`credit_value` & `transformed_credit`**: Display a consistent distribution scale lower than total property prices, with strong mutual correlation and moderate upper outliers.
* **`rent_value`**: Features wide interquartile variance extending significantly toward near-zero values, indicating "full deposit" (Rahn-e Kamel) listings rather than typical rental rates.
* **Daily Rent Columns (`rent_price_on_*`)**: Align at the lowest financial scale with sparse, discrete outliers, representing the most stable numeric distributions.

#### Actionable Next Steps
1. **Outlier Filtering**: Apply Interquartile Range (IQR) filtering or percentile clipping (e.g., trimming above the 99th percentile) to eliminate invalid extreme values.
2. **Zero-Value Treatment**: Explicitly flag listings with zero or nominal `rent_value` as full-deposit deals to prevent bias in rental regression models.
3. **Log Transformation**: Apply logarithmic scaling ($\log(1+x)$) across all target and financial features to normalize skewed distributions for linear and tree-based models.


In [ ]:
# Define corresponding categorical mode and numerical value pairs for financial features
pairs = [
    ("price_mode", "price_value"),
    ("rent_mode", "rent_value"),
    ("credit_mode", "credit_value"),
]

# Iterate through each pair to evaluate data integrity and cross-column consistency
for mode, val in pairs:
    # Ensure both columns exist in the DataFrame prior to evaluation
    if mode in df.columns and val in df.columns:
        # Create a boolean mask to identify discrepancies:
        # 1. Mode column is populated (not null and not just empty whitespace)
        # 2. Corresponding numeric value column is missing (null/NaN)
        mask = (
            df[mode].notna() & (df[mode].astype(str).str.strip() != "") & df[val].isna()
        )

        # Print the total count of affected records
        print(
            f"Count of records with '{mode}' filled but '{val}' missing: {mask.sum():,}"
        )

In [ ]:
for mode, val in pairs:
    if mode in df.columns and val in df.columns:
        mask = (
            df[mode].notna() & (df[mode].astype(str).str.strip() != "") & df[val].isna()
        )
        print(f"--- Unique values in '{mode}' when '{val}' is missing ---")
        print(df.loc[mask, mode].value_counts())
        print()

### "Agreed Price" (توافقی) Listings Validation

Data validation across `mode` and `value` column pairs confirms that:
* 100% of the non-empty `mode` records with missing numeric values (`value is NaN`) correspond strictly to the category **"توافقی" (Agreed / Negotiable)**.
* **Counts:**
  * `price_mode` = "توافقی" (Missing `price_value`): **5,260 listings**
  * `rent_mode` = "توافقی" (Missing `rent_value`): **1,672 listings**
  * `credit_mode` = "توافقی" (Missing `credit_value`): **899 listings**

#### Business & Modeling Decision:
* **For Price Prediction / Regression Models**: Exclude these records from training/testing datasets because the continuous target variable (`y`) is missing by design and cannot be imputed accurately.
* **For Market Analysis / Categorical Tasks**: These listings can be preserved to analyze the proportion and geographic distribution of "negotiable price" properties.


In [ ]:
# Cross-column logic and financial consistency checks

# 1. Check for zero or negative values in primary financial fields
invalid_zeros = {
    "price_zero_or_negative": (df["price_value"] <= 0).sum(),
    "credit_zero_or_negative": (df["credit_value"] <= 0).sum(),
    "rent_zero_or_negative": (df["rent_value"] <= 0).sum(),
}

# 2. Check for overlapping listings (both sale price and rental terms populated)
conflict_sale_rent = (
    df["price_value"].notna() & (df["rent_value"].notna() | df["credit_value"].notna())
).sum()

# 3. Check rental market structures (pure rent vs. pure deposit/credit)
pure_rent = (df["credit_value"].isna() & df["rent_value"].notna()).sum()
pure_credit = (df["credit_value"].notna() & df["rent_value"].isna()).sum()

# 4. Check daily rental sanity (weekend price lower than weekday price)
weekend_cheaper = (
    df["rent_price_at_weekends"] < df["rent_price_on_regular_days"]
).sum()

# Print results
print("--- Financial Consistency & Sanity Check Results ---")
print("Zero or negative values:", invalid_zeros)
print(
    f"Conflicting listings (Sale + Rent present simultaneously): {conflict_sale_rent:,}"
)
print(
    f"Pure rent (No deposit): {pure_rent:,} | Full credit (No monthly rent): {pure_credit:,}"
)
print(f"Inverted daily pricing (Weekend cheaper than regular day): {weekend_cheaper:,}")


#### Key Findings & Observations
1. **Zero & Negative Value Anomalies**:
   * **`rent_value <= 0` (59,241 records)**: Represents legitimate "Full Deposit / Full Mortgage" (*رهن کامل*) contracts where monthly rent is set to zero.
   * **`price_value <= 0` (1,902 records) & `credit_value <= 0` (2,864 records)**: Primarily invalid dummy values or negotiable/unspecified amounts recorded numerically as `0`.
2. **Strict Transaction Layer Isolation**:
   * **`conflict_sale_rent == 0`**: Confirms complete structural separation between the Sales market and Rental market pipelines. No listing simultaneously populates both sale and rental values.
3. **Niche Rental Structures**:
   * **Pure Rent (130 listings)**: Rare contract type requiring monthly rent without a deposit.
   * **Full Credit / No Monthly Rent (903 listings with NaN rent)**: Structural equivalent to the 59,241 zero-rent listings, represented as `NaN` rather than `0`.
4. **Daily Pricing Inversion**:
   * **292 listings** exhibit inverted weekend pricing (`rent_price_at_weekends < rent_price_on_regular_days`), pointing to manual input errors or non-standard dynamic pricing rules.

---

#### Engineering Decisions & Pipeline Rules
* **Segmentation**: Maintain separate modeling pipelines for **Sales** vs. **Rentals** given zero feature collision.
* **Mortgage Flagging**: Transform `rent_value == 0` records into a distinct boolean indicator (`is_full_mortgage = True`) rather than treating them as numerical missing values.
* **Cleaning & Filtering**: Drop non-positive `price_value` records ($N = 1,902$) from price prediction training sets to prevent regression distortion.


In [ ]:
# List of categorical features related to rental rules and conditions
cat_cols = ["rent_type", "rent_to_single", "rent_credit_transform"]

# Inspect unique category distributions and missingness (including NaN values)
for col in cat_cols:
    print(f"\n--- {col} ---")
    # Setting dropna=False is crucial here to capture the true volume of unpopulated/missing records
    print(df[col].value_counts(dropna=False))

### Rental Contract Categorical Features Analysis

To evaluate the informational value and data completeness of rental policy indicators, frequency distributions are computed for each category while explicitly preserving missing values (`dropna=False`).

#### Target Features
* **`rent_type`**: Contract pricing structure (e.g., standard rent & credit vs. full credit/deposit).
* **`rent_to_single`**: Policy flag indicating eligibility for single tenants.
* **`rent_credit_transform`**: Boolean flag indicating whether rent and deposit amounts are mutually negotiable/convertible.

#### Objectives
1. Quantify the exact missingness pattern across rental policy attributes.
2. Detect extreme class imbalance or quasi-zero variance columns that offer no predictive utility.
3. Determine whether qualitative contract tags align logically with numerical financial fields (e.g., zero rent listings).
